# Y-mas Tier 2 — Colab 학습 진입점

이 노트북은 **코드를 담지 않습니다.** 코드는 GitHub 래포가 단일 진실 소스이며,
여기서는 래포를 clone 하고 캐시를 연결해 학습만 돌립니다.

실행 순서: (1) 래포 clone → (2) Drive 마운트 → (3) 학습 → (4) ONNX 내보내기


## 1. 래포 clone + 의존성

In [ ]:
!git clone https://github.com/junparka-dotcom/Y-mas.git
%cd Y-mas
!git checkout refactor/v17-baseline   # 병합 후에는 main 사용
!pip install -q -r requirements.txt


## 2. Drive 마운트 (캐시 위치 연결)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 캐시 경로 -- 본인 Drive 구조에 맞게 수정
NTU  = '/content/drive/MyDrive/Action Recognition Dataset/cache/ntu_ymas_v15_T64.npz'
ETRI = '/content/drive/MyDrive/Action Recognition Dataset/cache/etri_ymas_v15_T64.npz'
OUT  = '/content/drive/MyDrive/Action Recognition Dataset/ckpt/ymas_v17_repro.pt'

import os
assert os.path.exists(NTU),  f'NTU 캐시 없음: {NTU}'
assert os.path.exists(ETRI), f'ETRI 캐시 없음: {ETRI}'
print('캐시 확인 완료')


## 3. v17 재현 학습

40 epoch 전체. 스모크로 빠르게 확인하려면 `--epochs 2` 추가.

In [ ]:
!python scripts/train_baseline.py --ntu "$NTU" --etri "$ETRI" --out "$OUT"


## 4. ONNX 내보내기 (Jetson 배포용)

`ymas_v17.onnx` (+ `.data`) 생성. pmean/pstd 도 함께 출력되니 추론 코드에 반영.

In [ ]:
ONNX = '/content/drive/MyDrive/Action Recognition Dataset/ckpt/ymas_v17_repro.onnx'
!python scripts/export_onnx.py --ckpt "$OUT" --out "$ONNX"


---
### 참고
- 학습은 GPU 런타임에서 실행하세요 (런타임 → 런타임 유형 변경 → T4 GPU).
- 캐시가 있으면 원본 NTU/ETRI zip(수십 GB) 파싱이 불필요합니다.
- 개선 실험은 `configs/` 에 새 config 를 만들어 진행합니다 (Phase 1).
